# Lib imports

In [1]:
from dotenv import load_dotenv

In [2]:
load_dotenv(override=True)

True

In [ ]:
import autoroot  # noqa
import nest_asyncio

# Third-party imports
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import PropertyGraphIndex, StorageContext
from llama_index.core.indices.property_graph import DynamicLLMPathExtractor
from llama_index.core.node_parser import SentenceSplitter
from llama_index.graph_stores.memgraph import MemgraphPropertyGraphStore
from llama_index.llms.openai import OpenAI
from llama_index.readers.wikipedia import WikipediaReader
from rich import print as rprint

import ogre_kg.kg_processors.entity_disambiguation as kg_disamb
import ogre_kg.kg_processors.entity_merger as kg_merger
import ogre_kg.kg_processors.entity_similarity_finders as kg_sim
import ogre_kg.kg_processors.graph_db_utils as kg_db_utils
import ogre_kg.kg_processors.retrievers.memgraph_retrievers as mg_ret

In [ ]:
nest_asyncio.apply()

## OGRE-KG modules overview

This notebook exercises the core OGRE-KG pipeline. The modules imported above cover four areas:

| Module | Purpose |
|--------|---------|
| `ogre_kg.kg_processors.graph_db_utils` | Builds full-text / vector indices in the graph backend so keyword retrievers can run efficiently. |
| `ogre_kg.kg_processors.entity_similarity_finders` | Finds groups of *similar* entity nodes using cosine similarity on stored embeddings (Memgraph backend) or fuzzy string matching (backend-agnostic). |
| `ogre_kg.kg_processors.entity_disambiguation` | Composes a similarity finder + a merger into a single `process()` call that runs the full find-then-merge pipeline. |
| `ogre_kg.kg_processors.entity_merger` | Applies the actual merge/synonym-creation strategy in the graph store (e.g. Memgraph `refactor.merge_nodes`, or Neo4j APOC). |
| `ogre_kg.kg_processors.retrievers.memgraph_retrievers` | Custom LlamaIndex retrievers that use Memgraph full-text search to seed context retrieval from the knowledge graph. |

# LLM config

In [4]:
llm = OpenAI(model="gpt-5-mini")
embedding = OpenAIEmbedding(model_name="text-embedding-3-small")

# Data prep

In [5]:
reader = WikipediaReader()
documents = reader.load_data(pages=["deep learning", "artificial inteligence"])

In [6]:
len(documents)

2

In [7]:
splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=64)
chunks = splitter.get_nodes_from_documents(documents)

In [8]:
rprint(chunks[0].get_content())

In machine learning, deep learning (DL) focuses on utilizing multilayered neural networks to perform tasks such as 
classification, regression, and representation learning. The field takes inspiration from biological neuroscience 
and revolves around stacking artificial neurons into layers and "training" them to process data. The adjective 
"deep" refers to the use of multiple layers (ranging from three to several hundred or thousands) in the network. 
Methods used can be supervised, semi-supervised or unsupervised.
Some common deep learning network architectures include fully connected networks, deep belief networks, recurrent 
neural networks, convolutional neural networks, generative adversarial networks, transformers, and neural radiance 
fields. These architectures have been applied to fields including computer vision, speech recognition, natural 
language processing, machine translation, bioinformatics, drug design, medical image analysis, climate science, 
material inspection and board game programs, where they have produced results comparable to and in some cases 
surpassing human expert performance.
Early forms of neural networks were inspired by information processing and distributed communication nodes in 
biological systems, particularly the human brain. However, current neural networks do not intend to model the brain
function of organisms, and are generally seen as low-quality models for that purpose.


== Overview ==
Most modern deep learning models are based on multi-layered neural networks such as convolutional neural networks 
and transformers, although they can also include propositional formulas or latent variables organized layer-wise in
deep generative models such as the nodes in deep belief networks and deep Boltzmann machines.
Fundamentally, deep learning refers to a class of machine learning algorithms in which a hierarchy of layers is 
used to transform input data into a progressively more abstract and composite representation. For example, in an 
image recognition model, the raw input may be an image (represented as a tensor of pixels). The first 
representational layer may attempt to identify basic shapes such as lines and circles, the second layer may compose
and encode arrangements of edges, the third layer may encode a nose and eyes, and the fourth layer may recognize 
that the image contains a face.
Importantly, a deep learning process can learn which features to optimally place at which level on its own. Prior 
to deep learning, machine learning techniques often involved hand-crafted feature engineering to transform the data
into a more suitable representation for a classification algorithm to operate on. In the deep learning approach, 
features are not hand-crafted and the model discovers useful feature representations from the data automatically. 
This does not eliminate the need for hand-tuning; for example, varying numbers of layers and layer sizes can 
provide different degrees of abstraction.
The word "deep" in "deep learning" refers to the number of layers through which the data is transformed. More 
precisely, deep learning systems have a substantial credit assignment path (CAP) depth. The CAP is the chain of 
transformations from input to output. CAPs describe potentially causal connections between input and output. For a 
feedforward neural network, the depth of the CAPs is that of the network and is the number of hidden layers plus 
one (as the output layer is also parameterized). For recurrent neural networks, in which a signal may propagate 
through a layer more than once, the CAP depth is potentially unlimited. No universally agreed-upon threshold of 
depth divides shallow learning from deep learning, but most researchers agree that deep learning involves CAP depth
higher than two. CAP of depth two has been shown to be a universal approximator in the sense that it can emulate 
any function. Beyond that, more layers do not add to the function approximator ability of the network. De

# Graph store prep

In [9]:
graph_store = MemgraphPropertyGraphStore(
    username="",
    password="",
    url="bolt://localhost:7688",
    database="memgraph",
)

graph_store.structured_query("MATCH (n) RETURN COUNT(n);")

[{'COUNT(n)': 312}]

In [10]:
recreate_index = False
extractor = DynamicLLMPathExtractor(llm=llm, max_triplets_per_chunk=5, num_workers=5)

if recreate_index:
    graph_store.structured_query("MATCH (n) DETACH DELETE n;")

    storage_ctx = StorageContext.from_defaults(property_graph_store=graph_store)
    index = PropertyGraphIndex(
        nodes=chunks,
        embed_model=embedding,
        kg_extractors=[extractor],
        storage_context=storage_ctx,
        show_progress=True,
    )
    index.storage_context.persist("./index/")
else:
    index = PropertyGraphIndex.from_existing(
        property_graph_store=graph_store,
        embed_model=embedding,
        kg_extractors=[extractor],
        show_progress=False,
    )

# Indexing

### Building full-text indices with `graph_db_utils`

`make_graph_text_index_builder(graph_store)` returns a backend-specific `GraphTextIndexBuilder`.  
Calling `create_indices()` creates two full-text indices (if they don't already exist):
- **`entity_name`** on `__Entity__.name` — used by keyword entity retrievers to seed graph traversal.
- **`chunk_text`** on `Chunk.text` — used by chunk-keyword retrievers to find relevant source text chunks.

These indices are **required** before using any `MemgraphKeywordContextRetriever` or `MemgraphChunkKeywordRetriever`.

In [11]:
builder = kg_db_utils.make_graph_text_index_builder(graph_store)
builder.create_indices()

# Similar entities finding

### Finding similar entities with `MemgraphCypherEntitySimilarityFinder`

`MemgraphCypherEntitySimilarityFinder` leverages Memgraph's built-in `node_similarity.cosine` procedure.  
It projects a subgraph of `__Entity__` nodes and their relationships, computes pairwise cosine similarity on the pre-stored `embedding` property, and returns pairs above the configured `similarity_threshold`.

The raw pairs are then grouped into **connected components** (union-find) so that transitively similar entities end up in the same set — this avoids creating redundant, partially overlapping groups in later merging steps.

In [12]:
sim_finder = kg_sim.MemgraphCypherEntitySimilarityFinder(
    graph_store, embedding_attr="embedding", similarity_threshold=0.95
)
similar_ents = sim_finder.find_similar_entities()
len(similar_ents)

0

In [65]:
similar_ents

[{'Generative adversarial network', 'Generative adversarial networks'},
 {'Open-weight model', 'Open-weight models'},
 {'Neural networks', 'neural networks'},
 {'Deep neural network', 'Deep neural networks'},
 {'Artificial neural network', 'Artificial neural networks'},
 {'Convolutional neural network', 'convolutional neural networks'},
 {'Central processing units', 'central processing units'}]

### Merging duplicates with `MemgraphEntityMerger` + `EntityDisambiguationProcessor`

Once similar-entity groups are known, two OGRE-KG components handle the actual graph modification:

- **`MemgraphEntityMerger`** — wraps Memgraph's `refactor.merge_nodes` MAGE procedure.  
  It accepts a `merging_strategy` (`override`, `combine`, or `discard`) that controls how conflicting node properties are resolved, and a `merge_relations` flag to also consolidate duplicate relationships.  
  Setting `preview_changes=True` lets you inspect the generated Cypher queries without touching the graph.

- **`EntityDisambiguationProcessor`** — acts as a *coordinator*: it calls `similarity_finder.find_similar_entities()` and pipes the result directly to `merger.merge_entities()`.  
  This separation of concerns means you can swap in any other finder (e.g. `FuzzyEntitySimilarityFinder`) or merger (e.g. `MemgraphSynonymCreator`, `Neo4jEntityMerger`) without changing the orchestration logic.

In [66]:
merger = kg_merger.MemgraphEntityMerger(graph_store, preview_changes=False)
disambiguator = kg_disamb.EntityDisambiguationProcessor(sim_finder, merger)

merged_ents = disambiguator.process()

# Extraction

### Custom retrievers from `memgraph_retrievers`

OGRE-KG provides two LlamaIndex-compatible sub-retrievers that use Memgraph full-text search:

- **`MemgraphKeywordContextRetriever`** — extracts keywords from the query via the LLM, then searches the `entity_name` full-text index to find matching entity nodes. From each seed entity it walks outgoing relationships in the graph and returns the surrounding context as `NodeWithScore` objects.

- **`MemgraphChunkKeywordRetriever`** — searches the `chunk_text` full-text index for document chunks whose text matches query keywords. When `restrict_to_seed_chunks=True` it limits traversal to chunks already linked to the seed entities found by the keyword search, keeping results tightly focused.

Both retrievers plug directly into a `PropertyGraphIndex.as_query_engine(sub_retrievers=[...])` call, replacing (or supplementing) the default vector and synonym retrievers supplied by LlamaIndex.

In [13]:
memgraph_kw_retriever = mg_ret.MemgraphKeywordContextRetriever(
    graph_store,
    llm=llm,
    topk_search=3,
    higher_score_is_better=True,
)
chunk_search = mg_ret.MemgraphChunkKeywordRetriever(
    graph_store,
    llm=llm,
    topk_search=3,
    max_chunk_terms=2,
    restrict_to_seed_chunks=True,
    higher_score_is_better=True,
)

In [14]:
question = "What are deep neural networks?"

In [15]:
kw_nodes = memgraph_kw_retriever.retrieve(question)
chunk_nodes = chunk_search.retrieve(question)

In [16]:
len(kw_nodes), len(chunk_nodes)

(11, 3)

In [17]:
rprint(kw_nodes[0].get_content())

Here are some facts extracted from the provided text:

Feedforward neural network -> TYPE_OF -> Artificial neural networks

== History ==


=== Before 1980 ===
There are two types of artificial neural network (ANN): feedforward neural network (FNN) or multilayer perceptron 
(MLP) and recurrent neural networks (RNN). RNNs have cycles in their connectivity structure, FNNs don't. In the 
1920s, Wilhelm Lenz and Ernst Ising created the Ising model which is essentially a non-learning RNN architecture 
consisting of neuron-like threshold elements. In 1972, Shun'ichi Amari made this architecture adaptive. His 
learning RNN was republished by John Hopfield in 1982. Other early recurrent neural networks were published by 
Kaoru Nakano in 1971. Already in 1948, Alan Turing produced work on "Intelligent Machinery"  that was not published
in his lifetime, containing "ideas related to artificial evolution and learning RNNs".
Frank Rosenblatt (1958) proposed the perceptron, an MLP with 3 layers: an input layer, a hidden layer with 
randomized weights that did not learn, and an output layer. He later published a 1962 book that also introduced 
variants and computer experiments, including a version with four-layer perceptrons "with adaptive preterminal 
networks" where the last two layers have learned weights (here he credits H. D. Block and B. W. Knight). The book 
cites an earlier network by R. D. Joseph (1960) "functionally equivalent to a variation of" this four-layer system 
(the book mentions Joseph over 30 times). Should Joseph therefore be considered the originator of proper adaptive 
multilayer perceptrons with learning hidden units? Unfortunately, the learning algorithm was not a functional one, 
and fell into oblivion.
The first working deep learning algorithm was the Group method of data handling, a method to train arbitrarily deep
neural networks, published by Alexey Ivakhnenko and Lapa in 1965. They regarded it as a form of polynomial 
regression, or a generalization of Rosenblatt's perceptron to handle more complex, nonlinear, and hierarchical 
relationships. A 1971 paper described a deep network with eight layers trained by this method, which is based on 
layer by layer training through regression analysis. Superfluous hidden units are pruned using a separate 
validation set. Since the activation functions of the nodes are Kolmogorov-Gabor polynomials, these were also the 
first deep networks with multiplicative units or "gates".
The first deep learning multilayer perceptron trained by stochastic gradient descent was published in 1967 by 
Shun'ichi Amari. In computer experiments conducted by Amari's student Saito, a five layer MLP with two modifiable 
layers learned  internal representations to classify non-linearily separable pattern classes. Subsequent 
developments in hardware and hyperparameter tunings have made end-to-end stochastic gradient descent the currently 
dominant training technique.
In 1969, Kunihiko Fukushima introduced the ReLU (rectified linear unit) activation function. The rectifier has 
become the most popular activation function for deep learning.
Deep learning architectures for convolutional neural networks (CNNs) with convolutional layers and downsampling 
layers began with the Neocognitron introduced by Kunihiko Fukushima in 1979, though not trained by backpropagation.
Backpropagation is an efficient application of the chain rule derived by Gottfried Wilhelm Leibniz in 1673 to 
networks of differentiable nodes. The terminology "back-propagating errors" was actually introduced in 1962 by 
Rosenblatt, but he did not know how to implement this, although Henry J. Kelley had a continuous precursor of 
backpropagation in 1960 in the context of control theory. The modern form of backpropagation was first published in
Seppo Linnainmaa's master thesis (1970). G.M. Ostrovski et al. republished it in 1971. Paul Werbos applied 
backpropagation to neural networks in 1982 (his 1974 PhD thesis, reprinted in a 1994 bo

In [20]:
qe = index.as_query_engine(sub_retrievers=[chunk_search, memgraph_kw_retriever], llm=llm)

In [21]:
res = qe.query(question)

In [23]:
rprint(res.response)

A deep neural network is an artificial neural network with multiple stacked layers of artificial neurons (i.e., a 
multilayered ANN). Its depth (more than a few layers) lets it learn complex, highly nonlinear relationships between
inputs and outputs, enabling tasks such as classification, regression and representation learning. These models are
usually trained end-to-end with gradient-based methods (e.g., backpropagation). Because of their large capacity 
they can approximate complicated functions but are also prone to overfitting and can be hard to interpret.